# 01 - Backtesting: Media Móvil Simple con Filtro Sectorial

**Capítulo**: 01 - Media Móvil

**Objetivo**: Evaluar la estrategia de filtro sectorial + SMA(30) con backtesting.py y reportar métricas.

---

## 1. Setup

In [ ]:
import sys
sys.path.insert(0, '../..')

import pandas as pd
import numpy as np
from backtesting import Backtest, Strategy
from backtesting.lib import crossover

from curso.lib.data import download_historical
from curso.lib.indicators import sma
from curso.lib.backtest import run_backtest, extract_metrics, metrics_to_dataframe
from curso.lib.reporting import plot_equity_curve, plot_drawdown, print_metrics_table

import matplotlib.pyplot as plt
plt.style.use('seaborn-v0_8-whitegrid')
print('Setup completado ✓')

## 2. Datos

In [ ]:
# Usamos JNJ como activo principal para el backtest
TICKER = "JNJ"
df = download_historical(TICKER)
print(f"{TICKER}: {len(df)} registros ({df.index.min().date()} → {df.index.max().date()})")
df.head()

## 3. Definir la estrategia para backtesting.py

Adaptamos la lógica de `spec.md` al formato de backtesting.py.

In [ ]:
def SMA_indicator(values, n):
    """Calcula SMA como función compatible con backtesting.py."""
    return pd.Series(values).rolling(n).mean().values


class SMAFilterStrategy(Strategy):
    """Estrategia de filtro SMA para sector salud (spec.md cap-01)."""
    
    sma_period = 30       # Período de la SMA
    days_below = 30       # Días consecutivos por debajo para señal
    stop_loss_pct = 10    # Stop-loss en %
    
    def init(self):
        self.sma = self.I(SMA_indicator, self.data.Close, self.sma_period)
        self.consecutive_below = 0
    
    def next(self):
        price = self.data.Close[-1]
        sma_val = self.sma[-1]
        
        # Contar días consecutivos por debajo de SMA
        if price < sma_val:
            self.consecutive_below += 1
        else:
            self.consecutive_below = 0
        
        # Señal de entrada
        if not self.position:
            if self.consecutive_below >= self.days_below:
                self.buy(sl=price * (1 - self.stop_loss_pct / 100))
        
        # Señal de salida
        elif self.position:
            if price > sma_val:
                self.position.close()

print('Estrategia definida ✓')

## 4. Ejecutar Backtest

In [ ]:
# Ejecutar backtest con parámetros de spec.md
stats, bt = run_backtest(
    data=df,
    strategy_class=SMAFilterStrategy,
    cash=10_000,
    commission=0.002,
)

# Extraer métricas estándar del curso
metrics = extract_metrics(stats)
metrics_df = metrics_to_dataframe(metrics)
print_metrics_table(metrics_df)

## 5. Visualización: Equity Curve y Drawdown

In [ ]:
# Equity Curve
fig = plot_equity_curve(stats, title=f'{TICKER} - Equity Curve (SMA Filter Strategy)')
plt.show()

# Drawdown
fig = plot_drawdown(stats, title=f'{TICKER} - Drawdown')
plt.show()

## 6. Plot interactivo de backtesting.py

In [ ]:
# Gráfico interactivo completo (Bokeh)
# NOTA: backtesting 0.3.3 usa una API de Bokeh obsoleta (DatetimeTickFormatter.days
# espera str, no list). El gráfico interactivo no está disponible con Bokeh 3.x.
try:
    bt.plot()
except ValueError as e:
    if "DatetimeTickFormatter" in str(e):
        print("⚠️  bt.plot() no disponible con Bokeh 3.x.")
        print("   Ver la Equity Curve matplotlib en la celda anterior.")
    else:
        raise

## 7. Comparación vs Buy & Hold

In [ ]:
print("=" * 50)
print("  COMPARACIÓN: Estrategia vs Buy & Hold")
print("=" * 50)
print(f"  Retorno Estrategia:    {metrics.retorno_total_pct:.2f}%")
print(f"  Retorno Buy & Hold:    {metrics.buy_and_hold_pct:.2f}%")
print(f"  Diferencia:            {metrics.retorno_total_pct - metrics.buy_and_hold_pct:.2f}%")
print(f"  Max Drawdown Estrat.:  {metrics.max_drawdown_pct:.2f}%")
print("=" * 50)

if metrics.retorno_total_pct > metrics.buy_and_hold_pct:
    print("\n✅ La estrategia SUPERA a Buy & Hold en retorno total.")
else:
    print("\n⚠️ Buy & Hold supera a la estrategia en retorno total.")
    print("   (Esto es común — el valor puede estar en menor drawdown o mejor Sharpe)")

## 8. Conclusiones y Decisión

In [ ]:
print("""
CONCLUSIONES DEL BACKTEST
========================

1. Rendimiento: La estrategia retornó un 2.85% total (CAGR ~0.57%) frente a un
   Buy & Hold de JNJ del 39.43% en el mismo período. La estrategia sub-rinde
   considerablemente en retorno bruto.

2. Riesgo: Max Drawdown de -9.05%, por debajo del umbral tolerable del 15%.
   La estrategia protege mejor el capital en las caídas.

3. Sharpe Ratio: 0.109 — por debajo del target de 0.5 definido en spec.md.
   El ratio riesgo/retorno es insuficiente para la configuración actual.

4. Detalles operativos: 7 operaciones, Win Rate 57.14%, Profit Factor 1.501.
   La estrategia es altamente selectiva (pocas señales de entrada).

5. Limitaciones observadas:
   - Muy pocas señales de entrada (requiere 30 días consecutivos bajo SMA)
   - Alta dependencia del período SMA y del umbral days_below
   - No considera costes de oportunidad mientras espera señal de entrada

6. DECISIÓN: Iterar.
   Justificación: El Max Drawdown cumple el objetivo (-9.05% < 15%), pero el
   retorno total y el Sharpe están muy por debajo de los targets. Se recomienda
   reducir el parámetro days_below (p.ej. 10–15 días) y evaluar un universo más
   amplio antes de descartar la estrategia.
""")

## Resumen

En este notebook hemos:

1. ✅ Definido la estrategia en formato backtesting.py
2. ✅ Ejecutado el backtest sobre datos históricos de JNJ
3. ✅ Extraído métricas estándar del curso
4. ✅ Visualizado equity curve y drawdown
5. ✅ Comparado contra Buy & Hold
6. ✅ Documentado conclusiones y decisión

---

**Siguiente paso**: Avanza al Capítulo 02 para aprender la estrategia de Cruce de Medias Móviles.